# DIRAC: VISUAL module overview

*Gosia Olejniczak, Warsaw University, 2025*

The VISUAL module in DIRAC enables the calculation of various scalar, vector, and tensor fields representing unperturbed and perturbed densities.

This notebook demonstrates how to integrate and/or visualize these densities on custom grids for a few simple molecules. Note that we use a poor setup (e.g., small basis sets, coarse grids) for demonstration purposes in all examples. For production calculations, the computational model needs to be well calibrated.

The calculations presented in this notebook can also be invoked by running the `visual_tutorial_overview` test (`ctest -R visual_tutorial_overview`) in the `DIRAC` automatic test system.

Here, we use the `py3Dmol` library for visualization and [RDKit](https://www.rdkit.org/) for molecular data manipulation, but any visualization software for plotting volumetric data can be used. We also demonstrate how to use different file formats, including HDF5, for which the `h5py` library is required. 
Please make sure that you have all required libraries and that DIRAC is successfully installed, and `pam` script is available (see the "Prerequisites" section below). 

### Prerequisites: environment preparation

If you are using `conda` package manager, then you can create a working environment as follows:

```
cd conda_envs
conda env create -f visual_overview.yml
cd ..
```

Make sure the environment is activated before running this notebook:

```
conda activate visual_overview
```

If successful, you should be able to execute the next cell without errors:

In [7]:
import os,sys
import re
from pathlib import Path
import shutil
import subprocess
from IPython.display import display, HTML

import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import IPythonConsole
Chem.Draw.IPythonConsole.ipython_3d = True
from rdkit import rdBase
from rdkit.Chem import rdmolfiles
from rdkit.Chem import rdDetermineBonds
from rdkit.Chem import rdMolAlign
from rdkit.Chem import rdMolTransforms

import py3Dmol

from ipywidgets import interact,fixed,IntSlider

Next, we set up paths to DIRAC `pam` script.

In this notebook, we assume the `pam` script is present in the `../build` directory. If you installed `DIRAC` in a different directory, adapt the paths as needed:

In [2]:
# DIRAC
notebook_path=Path().parent.resolve()
dirac_path=notebook_path.parent.resolve()
dirac_bin=Path(dirac_path, 'build').resolve()
dirac_testset=Path(dirac_path, 'test').resolve()
pam=Path(dirac_bin, 'pam')

def run_dirac(cmd, log=False):
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if log:
        print(result.stdout, end="")

Print out the path to `pam` used in this notebook:

In [3]:
cmd="which pam"
result=subprocess.run(cmd, shell=True, text=True, capture_output=True)
pam=result.stdout
print(pam)

/home/gosia/devel/dirac-public/build/pam



Finally, we set up the working directory for all files used or generated in this notebook:

In [4]:
test_name='visual_tutorial_overview'
workdir=Path(Path.cwd(), 'workdir', test_name).resolve()
workdir.mkdir(parents=True, exist_ok=True)
str(workdir)

'/home/gosia/devel/dirac-public/demo_notebooks/workdir/visual_tutorial_overview'

## Visualization of the electron density


In this example, we visualize and integrate the electron density of water molecule.

First, we prepare input files in the working directory. We use input files from the 'visual_tutorial_overview' test in DIRAC automatic test set:

In [5]:
os.chdir(str(workdir))
print('DIRAC will run in: %s' %(workdir))

DIRAC will run in: /home/gosia/devel/dirac-public/demo_notebooks/workdir/visual_tutorial_overview


In [6]:
dirac_test=Path(dirac_testset, test_name).resolve()
fmol=str(Path(dirac_test, 'h2o.xyz').resolve())
finp1=str(Path(dirac_test, 'dc.inp').resolve())
finp2=str(Path(dirac_test, 'vis.inp').resolve())

print('The input files are in the corresponding DIRAC test directory: %s' %(dirac_test))

The input files are in the corresponding DIRAC test directory: /home/gosia/devel/dirac-public/test/visual_tutorial_overview


Let us first look at the molecule:

In [8]:
view = py3Dmol.view(width=400, height=300)

def from_xyz_to_mol(fxyz):
    raw_mol = rdmolfiles.MolFromXYZFile(str(fxyz))
    mol = Chem.Mol(raw_mol)
    rdDetermineBonds.DetermineConnectivity(mol)
    return mol

mol=from_xyz_to_mol(fmol)
view = py3Dmol.view(
    data=Chem.MolToMolBlock(mol),
    style={"stick": {}, "sphere": {"scale": 0.3}},
    width=300,
    height=300,
)
view.zoomTo()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

We are now prepared to run calculations. 

Note that in this tutorial, we run calculations in two steps, but they can also be done as one job. First, we optimize the wave function and save the result on the 'CHECKPOINT.h5' file, for which we use the `--outcmo` option. 

In [ ]:
cmd = 'pam --inp=%s --mol=%s --noarch --nobackup --outcmo' % (finp1, fmol)
print('DIRAC run command (step 1): ')
print(cmd)
run_dirac(cmd)

Then, we do calculations in the VISUAL module, using the generated 'CHECKPOINT.h5' file (the `--incmo` option):

In [ ]:
cmd = 'pam --inp=%s --mol=%s --noarch --nobackup --incmo --get="ed.cube"' % (finp2, fmol)
print('DIRAC run command (step 2): ')
print(cmd)
run_dirac(cmd)

In this job, we export the electron density on a 3D rectilinear grid with 64x64x32 points in Gaussian `cube` format to the 'ed.cube' file, as declared in the first grid function specification. We also integrate the electron density on the same numerical grid as used in the DFT calculation in step 1, as declared in the second grid function specification:

In [15]:
greplines="sed -n '/VISUAL/,/END/p' %s " %(finp2)
out=subprocess.run(greplines, shell=True, text=True, capture_output=True)
print(out.stdout)

**VISUAL
.GRIDS 
2
id=1 input=create typ=1 npoints=[64,64,32] margin=4.0
id=2 input=dft
.GRIDFUNCTIONS
2
name=ed id_grid=1 purpose=visualization export=cube file_out=ed.cube
name=ed id_grid=2 purpose=integration
*END OF



Now, we can visualize the electron density on the 'ed.cube' file:

In [16]:
view = py3Dmol.view(width=400, height=300)

In [17]:
data = None
with open("ed.cube", "r") as infile:
    data = infile.read()

In [18]:
view = py3Dmol.view()
view.addVolumetricData(
    data,
    "cube",
    {
        "isoval": 0.1,
        "smoothness": 5,
        "opacity": 0.8,
        "volformat": "cube",
        "color": "blue",
    },
)
view.addModel(data, "cube")
view.setStyle({"stick": {}, "sphere": {"scale": 0.3}})
view.zoomTo()
view.show()


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

To inspect the result of integration, we look in the DIRAC output file:

In [32]:
fout=Path(workdir, finp2.split('/')[-1].strip('.inp')+'_'+fmol.split('/')[-1].strip('.xyz')+'.out').resolve()

greplines="sed -n '/Final grid_function values /,/^[^Elapsed]/p' %s " %(fout)
out=subprocess.run(greplines, shell=True, text=True, capture_output=True)
print(out.stdout)

Final grid_function values (grid_function%id=  2) integrated on grid (grid#id=  2): 
   9.99999977E+000



As expected, the integrated value is close to 10, which is the number of electrons in water molecule.

In [ ]:
os.chdir(str(notebook_path))

## Visualization of ELF in 3D

In this example, we visualize the electron localization function in benzene:

First, we prepare input files in the working directory. We use input files from the 'visual_tutorial_overview' test in DIRAC automatic test set:

## Visualization of orbital densities

## Visualization of the magnetically-induced current density


## The magnetizability density


## The NMR shielding density


## Radial distributions
